In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# 1. Загрузка данных (используем прямую ссылку на CSV из репозитория UCI)
url = "https://archive.ics.uci.edu"
df = pd.read_csv(url)

# Предварительная обработка: удаляем 'stabf', так как это категориальная копия целевой 'stab'
df = df.drop(columns=['stabf'])

# 2.1. EDA и базовая проверка
print(f"Формат данных: {df.shape}")
print(df.info())

# 2.2. Корреляционная матрица (12 признаков: tau1-tau4, p1-p4, g1-g4)
features = df.drop(columns=['stab'])
corr_matrix = df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', fmt='.3f')
plt.title("Корреляционная матрица Пирсона")
plt.show()

# Описание зависимостей (пример):
# "Наибольшая корреляция с целевой переменной 'stab' наблюдается у признаков группы 'g' (коэффициенты ~0.28).
# Между признаками tau, p и g корреляция практически отсутствует (близка к 0), что говорит о независимости входных параметров."

# 2.3. Оценка мультиколлинеарности (VIF)
vif_data = pd.DataFrame()
vif_data["feature"] = features.columns
vif_data["VIF"] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]
print("\nПоказатели VIF:\n", vif_data)

# 2.4. Построение моделей
X = features
y = df['stab']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Масштабирование (важно для регуляризации)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5)
}

# 2.5. Создание таблицы результатов
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    # Предсказания
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "Train R2": round(r2_score(y_train, train_pred), 4),
        "Test R2": round(r2_score(y_test, test_pred), 4),
        "Train MSE": round(mean_squared_error(y_train, train_pred), 6),
        "Test MSE": round(mean_squared_error(y_test, test_pred), 6)
    })

results_df = pd.DataFrame(results)
print("\nИтоговая таблица метрик:")
print(results_df)
